In [1]:
# ─────────── MODELS ───────────

import warnings
warnings.filterwarnings("ignore")

import re
import string
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score, roc_auc_score,
    ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight
import joblib


In [2]:
# ─────────── CONSTANTS ───────────

CLASS_NAMES = {0: "Hate Speech", 1: "Offensive", 2: "Neither"}
PALETTE = {"Hate Speech": "#E24B4A", "Offensive": "#EF9F27", "Neither": "#1D9E75"}
COLOR_LIST = [PALETTE["Hate Speech"], PALETTE["Offensive"], PALETTE["Neither"]]


In [3]:
# ─────────── DATA LOADING ───────────

print("=" * 60)
print("  OFFENSIVE LANGUAGE DETECTION — ML PIPELINE")
print("=" * 60)

df = pd.read_csv("/content/labeled_data.csv", index_col=0)
df["label_name"] = df["class"].map(CLASS_NAMES)

print(f"\n[1] Dataset loaded: {len(df):,} samples")
print(f"    Columns: {list(df.columns)}")
print(f"\n    Class distribution:")
for cls, name in CLASS_NAMES.items():
    n = (df["class"] == cls).sum()
    pct = n / len(df) * 100
    print(f"      {name:20s}: {n:5,}  ({pct:.1f}%)")


  OFFENSIVE LANGUAGE DETECTION — ML PIPELINE

[1] Dataset loaded: 6,500 samples
    Columns: ['count', 'hate_speech', 'offensive_language', 'neither', 'class', 'tweet', 'label_name']

    Class distribution:
      Hate Speech         :   380  (5.8%)
      Offensive           : 4,163  (64.0%)
      Neither             : 1,957  (30.1%)


In [4]:

# ───────────TEXT PREPROCESSING ───────────

STOPWORDS = set([
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your",
    "yours", "he", "him", "his", "she", "her", "hers", "it", "its", "they", "them",
    "their", "what", "which", "who", "whom", "this", "that", "these", "those",
    "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had",
    "do", "does", "did", "will", "would", "shall", "should", "may", "might", "must",
    "can", "could", "a", "an", "the", "and", "but", "if", "or", "because", "as",
    "until", "while", "of", "at", "by", "for", "with", "about", "against", "between",
    "into", "through", "during", "before", "after", "to", "from", "in", "out", "on",
    "off", "then", "so", "than", "too", "very", "just", "up", "down", "not", "no",
])

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)        # remove URLs
    text = re.sub(r"@\w+", "", text)                   # remove @mentions
    text = re.sub(r"#(\w+)", r"\1", text)              # keep hashtag words
    text = re.sub(r"rt\s+", "", text)                  # remove RT
    text = re.sub(r"[^\w\s]", " ", text)               # remove punctuation
    text = re.sub(r"\d+", "", text)                    # remove digits
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return " ".join(tokens)

print("\n[2] Preprocessing tweets...")
df["clean_tweet"] = df["tweet"].apply(preprocess)
df["tweet_length"] = df["tweet"].apply(len)
df["word_count"] = df["tweet"].apply(lambda x: len(x.split()))
df["exclamation_count"] = df["tweet"].apply(lambda x: x.count("!"))
df["caps_ratio"] = df["tweet"].apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1))

sample_tweet = df.iloc[0]["tweet"]
sample_clean = df.iloc[0]["clean_tweet"]
print(f"    Original : {sample_tweet[:80]}...")
print(f"    Cleaned  : {sample_clean[:80]}...")


[2] Preprocessing tweets...
    Original : What a stupid absolute move, I can't believe this crap wtf...
    Cleaned  : stupid absolute move believe crap wtf...


In [5]:
# ─────────── TRAIN / TEST SPLIT ───────────

X = df["clean_tweet"]
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n[3] Train/Test split: {len(X_train):,} train / {len(X_test):,} test")



[3] Train/Test split: 5,200 train / 1,300 test


In [6]:
# ─────────── MODELS ───────────

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=20000,
    sublinear_tf=True,
    min_df=2
)

class_weights = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=y_train)
cw_dict = {0: class_weights[0], 1: class_weights[1], 2: class_weights[2]}

models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=20000, sublinear_tf=True, min_df=2)),
        ("clf", LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000, random_state=42))
    ]),
    "Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=20000, sublinear_tf=True, min_df=2)),
        ("clf", LinearSVC(C=0.5, class_weight="balanced", max_iter=2000, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=10000, sublinear_tf=True, min_df=2)),
        ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=42))
    ]),
    "Complement NB": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=20000, sublinear_tf=True, min_df=2)),
        ("clf", ComplementNB(alpha=0.3))
    ]),
}

print("\n[4] Training models...\n")
results = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")
    f1_weighted = f1_score(y_test, y_pred, average="weighted")
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=list(CLASS_NAMES.values()), output_dict=True)

    results[name] = {
        "pipeline": pipeline,
        "y_pred": y_pred,
        "accuracy": acc,
        "f1_macro": f1_macro,   # F1 score for multi class classification
        "f1_weighted": f1_weighted,   # F1-score for each class, weighted
        "confusion_matrix": cm,
        "report": report,
    }
    print(f"  {name:25s} | Acc: {acc:.4f} | F1-macro: {f1_macro:.4f} | F1-weighted: {f1_weighted:.4f}")



[4] Training models...

  Logistic Regression       | Acc: 1.0000 | F1-macro: 1.0000 | F1-weighted: 1.0000
  Linear SVM                | Acc: 1.0000 | F1-macro: 1.0000 | F1-weighted: 1.0000
  Random Forest             | Acc: 1.0000 | F1-macro: 1.0000 | F1-weighted: 1.0000
  Complement NB             | Acc: 1.0000 | F1-macro: 1.0000 | F1-weighted: 1.0000


In [11]:
# ─────────── BEST MODEL ───────────

best_name = max(results, key=lambda K : results[K]["f1_macro"])
best = results[best_name]
print(f"\n[5] Best model: {best_name} (F1-macro: {best['f1_macro']:.4f})")
print(f"\n    Classification Report ({best_name}):\n")
print(classification_report(y_test, best["y_pred"], target_names=list(CLASS_NAMES.values())))

# Save best model
joblib.dump(best["pipeline"], "/content/best_model.pkl")
print(f"    Model saved to: /content/best_model.pkl")


[5] Best model: Logistic Regression (F1-macro: 1.0000)

    Classification Report (Logistic Regression):

              precision    recall  f1-score   support

 Hate Speech       1.00      1.00      1.00        76
   Offensive       1.00      1.00      1.00       833
     Neither       1.00      1.00      1.00       391

    accuracy                           1.00      1300
   macro avg       1.00      1.00      1.00      1300
weighted avg       1.00      1.00      1.00      1300

    Model saved to: /content/best_model.pkl


In [12]:
# ─────────── VISUALIZATION DASHBOARD ───────────

print("\n[6] Generating visualisation dashboard...")

plt.style.use("seaborn-v0_8-whitegrid")
FONT_COLOR = "#2C2C2A"
BG = "#FAFAFA"

fig = plt.figure(figsize=(20, 22), facecolor=BG)
fig.suptitle(
    "Offensive Language Detection — ML Analysis Dashboard",
    fontsize=20, fontweight="bold", color=FONT_COLOR, y=0.98
)
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.52, wspace=0.38)

# ── Panel 1: Class distribution (bar)
ax1 = fig.add_subplot(gs[0, 0])
class_counts = df["label_name"].value_counts()
bars = ax1.bar(class_counts.index, class_counts.values,
               color=[PALETTE[c] for c in class_counts.index],
               width=0.55, edgecolor="white", linewidth=0.8)
for bar, v in zip(bars, class_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
             f"{v:,}", ha="center", va="bottom", fontsize=10, fontweight="bold", color=FONT_COLOR)
ax1.set_title("Class Distribution", fontsize=13, fontweight="bold", color=FONT_COLOR, pad=10)
ax1.set_ylabel("Count", color=FONT_COLOR)
ax1.tick_params(colors=FONT_COLOR)
ax1.set_facecolor(BG)
ax1.spines[["top", "right"]].set_visible(False)

# ── Panel 2: Class distribution (pie)
ax2 = fig.add_subplot(gs[0, 1])
wedge_props = dict(linewidth=2, edgecolor="white")
wedges, texts, autotexts = ax2.pie(
    class_counts.values,
    labels=class_counts.index,
    colors=[PALETTE[c] for c in class_counts.index],
    autopct="%1.1f%%",
    startangle=140,
    wedgeprops=wedge_props,
    pctdistance=0.75,
)
for t in texts: t.set_fontsize(10); t.set_color(FONT_COLOR)
for at in autotexts: at.set_fontsize(9); at.set_color("white"); at.set_fontweight("bold")
ax2.set_title("Class Proportions", fontsize=13, fontweight="bold", color=FONT_COLOR, pad=10)

# ── Panel 3: Model accuracy comparison
ax3 = fig.add_subplot(gs[0, 2])
model_names = list(results.keys())
accs = [results[m]["accuracy"] for m in model_names]
f1s = [results[m]["f1_macro"] for m in model_names]
x = np.arange(len(model_names))
w = 0.35
bars1 = ax3.bar(x - w/2, accs, w, label="Accuracy", color="#378ADD", edgecolor="white", linewidth=0.8)
bars2 = ax3.bar(x + w/2, f1s, w, label="F1-macro", color="#1D9E75", edgecolor="white", linewidth=0.8)
ax3.set_xticks(x)
ax3.set_xticklabels([m.replace(" ", "\n") for m in model_names], fontsize=8, color=FONT_COLOR)
ax3.set_ylim(0, 1.05)
ax3.set_title("Model Comparison", fontsize=13, fontweight="bold", color=FONT_COLOR, pad=10)
ax3.set_ylabel("Score", color=FONT_COLOR)
ax3.legend(fontsize=9)
ax3.tick_params(colors=FONT_COLOR)
ax3.set_facecolor(BG)
ax3.spines[["top", "right"]].set_visible(False)
for b in list(bars1) + list(bars2):
    ax3.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
             f"{b.get_height():.2f}", ha="center", va="bottom", fontsize=7, color=FONT_COLOR)

# ── Panel 4: Best model confusion matrix
ax4 = fig.add_subplot(gs[1, 0:2])
cm = best["confusion_matrix"]
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(
    cm_pct, annot=True, fmt=".2f", cmap="YlOrRd",
    xticklabels=list(CLASS_NAMES.values()),
    yticklabels=list(CLASS_NAMES.values()),
    linewidths=0.5, linecolor="white",
    ax=ax4, cbar_kws={"shrink": 0.8}
)
# Overlay raw counts
for i in range(3):
    for j in range(3):
        ax4.text(j + 0.5, i + 0.72, f"n={cm[i,j]}", ha="center",
                 va="center", fontsize=8, color="gray")
ax4.set_title(f"Confusion Matrix — {best_name} (normalised)", fontsize=13, fontweight="bold", color=FONT_COLOR, pad=10)
ax4.set_ylabel("True Label", color=FONT_COLOR)
ax4.set_xlabel("Predicted Label", color=FONT_COLOR)
ax4.tick_params(colors=FONT_COLOR)

# ── Panel 5: Per-class F1 heatmap
ax5 = fig.add_subplot(gs[1, 2])
f1_data = []
for m in model_names:
    row = []
    for cls_name in CLASS_NAMES.values():
        row.append(results[m]["report"][cls_name]["f1-score"])
    f1_data.append(row)
f1_df = pd.DataFrame(f1_data, index=[m.replace(" ", "\n") for m in model_names],
                     columns=list(CLASS_NAMES.values()))
sns.heatmap(f1_df, annot=True, fmt=".2f", cmap="Greens",
            linewidths=0.5, linecolor="white",
            ax=ax5, cbar_kws={"shrink": 0.8}, vmin=0, vmax=1)
ax5.set_title("Per-class F1 by Model", fontsize=13, fontweight="bold", color=FONT_COLOR, pad=10)
ax5.tick_params(colors=FONT_COLOR, labelsize=8)

# ── Panel 6: Tweet length distribution
ax6 = fig.add_subplot(gs[2, 0])
for cls, name in CLASS_NAMES.items():
    subset = df[df["class"] == cls]["tweet_length"]
    ax6.hist(subset, bins=30, alpha=0.6, color=PALETTE[name], label=name, density=True)
ax6.set_title("Tweet Length Distribution", fontsize=13, fontweight="bold", color=FONT_COLOR, pad=10)
ax6.set_xlabel("Character count", color=FONT_COLOR)
ax6.set_ylabel("Density", color=FONT_COLOR)
ax6.legend(fontsize=9)
ax6.set_facecolor(BG)
ax6.spines[["top", "right"]].set_visible(False)
ax6.tick_params(colors=FONT_COLOR)

# ── Panel 7: Word count box plot
ax7 = fig.add_subplot(gs[2, 1])
bp_data = [df[df["class"] == cls]["word_count"].values for cls in CLASS_NAMES]
bp = ax7.boxplot(bp_data, patch_artist=True, notch=True,
                 medianprops=dict(color="white", linewidth=2))
for patch, color in zip(bp["boxes"], COLOR_LIST):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax7.set_xticklabels(list(CLASS_NAMES.values()), fontsize=9, color=FONT_COLOR)
ax7.set_title("Word Count by Class", fontsize=13, fontweight="bold", color=FONT_COLOR, pad=10)
ax7.set_ylabel("Words per tweet", color=FONT_COLOR)
ax7.set_facecolor(BG)
ax7.spines[["top", "right"]].set_visible(False)
ax7.tick_params(colors=FONT_COLOR)

# ── Panel 8: Top TF-IDF features per class
ax8 = fig.add_subplot(gs[2, 2])
best_tfidf = best["pipeline"].named_steps["tfidf"]
best_clf = best["pipeline"].named_steps["clf"]
feature_names = np.array(best_tfidf.get_feature_names_out())

# For LR/LinearSVC: get top features per class 0 (hate)
if hasattr(best_clf, "coef_"):
    coef = best_clf.coef_
    for row_idx, (cls, name) in enumerate(CLASS_NAMES.items()):
        if row_idx < coef.shape[0]:
            top_idx = np.argsort(coef[row_idx])[-8:]
            top_words = feature_names[top_idx]
            top_scores = coef[row_idx][top_idx]
            y_pos = np.arange(len(top_words)) + row_idx * 9
            ax8.barh(y_pos, top_scores, color=PALETTE[name], alpha=0.8, height=0.7)
            for i, (w, s) in enumerate(zip(top_words, top_scores)):
                ax8.text(s + 0.005, y_pos[i], w, va="center", fontsize=7, color=FONT_COLOR)
    ax8.set_title("Top TF-IDF Features\n(per class)", fontsize=11, fontweight="bold", color=FONT_COLOR, pad=6)
    ax8.axis("off")
else:
    ax8.text(0.5, 0.5, "Feature importance\nnot available\nfor this model",
             ha="center", va="center", transform=ax8.transAxes, fontsize=11, color="gray")
    ax8.set_title("Feature Importance", fontsize=13, fontweight="bold", color=FONT_COLOR, pad=10)

# ── Panel 9: Caps ratio & exclamation
ax9 = fig.add_subplot(gs[3, 0])
for cls, name in CLASS_NAMES.items():
    subset = df[df["class"] == cls]["caps_ratio"]
    ax9.hist(subset, bins=25, alpha=0.6, color=PALETTE[name], label=name, density=True)
ax9.set_title("CAPS Ratio Distribution", fontsize=13, fontweight="bold", color=FONT_COLOR, pad=10)
ax9.set_xlabel("Proportion of uppercase chars", color=FONT_COLOR)
ax9.set_ylabel("Density", color=FONT_COLOR)
ax9.legend(fontsize=9)
ax9.set_facecolor(BG)
ax9.spines[["top", "right"]].set_visible(False)
ax9.tick_params(colors=FONT_COLOR)

# ── Panel 10: Annotator agreement
ax10 = fig.add_subplot(gs[3, 1])
df["agreement"] = df.apply(
    lambda r: max(r["hate_speech"], r["offensive_language"], r["neither"]) / r["count"], axis=1
)
for cls, name in CLASS_NAMES.items():
    subset = df[df["class"] == cls]["agreement"]
    ax10.hist(subset, bins=20, alpha=0.6, color=PALETTE[name], label=name, density=True)
ax10.set_title("Annotator Agreement Distribution", fontsize=13, fontweight="bold", color=FONT_COLOR, pad=10)
ax10.set_xlabel("Agreement ratio (majority / total)", color=FONT_COLOR)
ax10.set_ylabel("Density", color=FONT_COLOR)
ax10.legend(fontsize=9)
ax10.set_facecolor(BG)
ax10.spines[["top", "right"]].set_visible(False)
ax10.tick_params(colors=FONT_COLOR)

# ── Panel 11: Summary metrics table
ax11 = fig.add_subplot(gs[3, 2])
ax11.axis("off")
table_data = []
for m in model_names:
    table_data.append([
        m,
        f"{results[m]['accuracy']:.3f}",
        f"{results[m]['f1_macro']:.3f}",
        f"{results[m]['f1_weighted']:.3f}",
    ])
headers = ["Model", "Accuracy", "F1-macro", "F1-weighted"]
tbl = ax11.table(
    cellText=table_data,
    colLabels=headers,
    loc="center",
    cellLoc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.1, 1.8)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("#D3D1C7")
    if row == 0:
        cell.set_facecolor("#2C2C2A")
        cell.set_text_props(color="white", fontweight="bold")
    else:
        m_name = model_names[row - 1]
        if m_name == best_name:
            cell.set_facecolor("#EAF3DE")
        else:
            cell.set_facecolor(BG if row % 2 == 0 else "#F5F5F2")

ax11.set_title(f"Results Summary\n(★ = best: {best_name})", fontsize=11, fontweight="bold",
               color=FONT_COLOR, pad=8)

plt.savefig("/content/dashboard.png", dpi=150, bbox_inches="tight",
            facecolor=BG, edgecolor="none")
print("    Dashboard saved to: dashboard.png")



[6] Generating visualisation dashboard...
    Dashboard saved to: dashboard.png


In [13]:
# ─────────── INFERENCE FUNCTION ───────────

def predict(text: str) -> dict:
    """
    Run the best model on a raw text string.
    Returns dict with label, label_name, confidence.
    """
    model = joblib.load("/content/best_model.pkl")
    clean = preprocess(text)
    label = model.predict([clean])[0]
    if hasattr(model.named_steps["clf"], "predict_proba"):
        proba = model.predict_proba([clean])[0]
        confidence = proba[label]
    else:
        confidence = None
    return {
        "text": text,
        "clean_text": clean,
        "label": int(label),
        "label_name": CLASS_NAMES[label],
        "confidence": round(float(confidence), 4) if confidence is not None else "N/A",
    }

print("\n[7] Inference demo:")
demo_texts = [
    "You're an absolute idiot, go screw yourself",
    "Had a wonderful time at the park today with my family",
    "All those people should be eliminated from our society",
]
for t in demo_texts:
    r = predict(t)
    print(f"  [{r['label_name']:20s}] conf={r['confidence']}  → \"{t[:60]}\"")

print("\n" + "=" * 60)
print("  Pipeline complete.")
print("  Outputs: best_model.pkl | dashboard.png")
print("=" * 60)




[7] Inference demo:
  [Offensive           ] conf=0.9761  → "You're an absolute idiot, go screw yourself"
  [Neither             ] conf=0.8544  → "Had a wonderful time at the park today with my family"
  [Hate Speech         ] conf=0.9811  → "All those people should be eliminated from our society"

  Pipeline complete.
  Outputs: best_model.pkl | dashboard.png


In [14]:
# ─────────── USER INPUT ───────────

user_input_text = input("Enter the text you want to classify: ")

if user_input_text:
    prediction_result = predict(user_input_text)
    print(f"\nClassification Result for: \"{user_input_text}\"")
    print(f"  Predicted Label: {prediction_result['label_name']}")
    print(f"  Confidence: {prediction_result['confidence']}")
    print(f"  Cleaned Text: {prediction_result['clean_text']}")
else:
    print("No text entered. Please provide some text to classify.")

Enter the text you want to classify: You're an absolute idiot, go screw yourself 

Classification Result for: "You're an absolute idiot, go screw yourself "
  Predicted Label: Offensive
  Confidence: 0.9761
  Cleaned Text: re absolute idiot go screw yourself
